In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import argparse
import numpy as np
import glob
from rich.progress import Progress, TextColumn, BarColumn, TaskProgressColumn
from time import time

import torchvision

In [3]:
def calculate_mean_std(**kwargs):
    """
    Fill in the per channel mean and standard deviation of the dataset. 
    Just fill in the values, no need to compute them.
    """
    return [0.5127, 0.4529, 0.3974], [0.2496, 0.2534, 0.2615]

In [4]:



# # return [mean_r, mean_g, mean_b], [std_r, std_g, std_b]
#     #1. list of images(not same size) 
#     # for each image, for each channel, add all the values and count all the values.
#     # at the end, for R, we have final sum and count. calculate mean
#     # do this for all color channels.
#     # for each image, for eachchannel, do sum((channel - mean)**2)
#     # sum for all images
#     # sum/
#     data_dir = "hw4-data"
#     images = []
#     channels_sum = torch.zeros(3)
#     channels_squared_sum = torch.zeros(3)
#     num_pixels = 0
#     transform = transforms.ToTensor()

#     for folder in os.listdir(data_dir):
#         folder_path = os.path.join(data_dir, folder)
#         if os.path.isdir(folder_path):
#             for subfolder in os.listdir(folder_path):
#                 sub_path = os.path.join(folder_path, subfolder)
#                 for img in os.listdir(sub_path):
#                     if img.lower().endswith(".jpg"):
#                         images.append(os.path.join(sub_path, img))
    
#     for image in images:
#         img = Image.open(image).convert("RGB")
#         image_tensor = transform(img)
#         num_pixels += image_tensor.size(1)*image_tensor.size(2)
#         channels_sum += torch.mean(image_tensor, dim= [1,2])*(image_tensor.size(1)*image_tensor.size(2))
#         channels_squared_sum += torch.mean(image_tensor**2, dim = [1,2])*(image_tensor.size(1) * image_tensor.size(2))

#     mean = channels_sum / num_pixels
#     std = torch.sqrt((channels_squared_sum / num_pixels) - (mean ** 2))
    
#     final_mean = [round(x.item(), 4) for x in mean]
#     final_std = [round(x.item(), 4) for x in std]

#     return final_mean, final_std

calculate_mean_std()


# dataset = SUN397Dataset("hw4-data", transform)

# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size

# train_ds, val_ds = random_split(dataset,[train_size,val_size])

# train_loader = DataLoader(train_ds,batch_size=64,shuffle=True)
# val_loader = DataLoader(val_ds,batch_size=64)
# torch.tensor(dataset.X)
# dataset.X[10].shape






([0.5127, 0.4529, 0.3974], [0.2496, 0.2534, 0.2615])

In [5]:
class SUN397Dataset(Dataset):
    """
    A custom dataset class for loading the SUN397 dataset.
    """

    def __init__(self, data_dir, transform=None):
        """
        Initializes the dataset with images and labels.
        Args:
            data_dir (str): Path to the data directory.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        self.data_dir = data_dir
        self.X = []
        self.y = []
        self.labels = []
        mean, std = calculate_mean_std()
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean, std)
            ])
        else:
            self.transform = transform

        for folder in os.listdir(self.data_dir):
            if os.path.isdir(os.path.join(self.data_dir, folder)):
                for subfolder in os.listdir(os.path.join(self.data_dir, folder)):
                    label = f"{folder}/{subfolder}"
                    for img in os.listdir(os.path.join(self.data_dir, folder, subfolder)):
                        if img.lower().endswith(".jpg"):
                            full_path = os.path.join(self.data_dir, folder, subfolder, img)
                            
                            self.X.append(full_path)
                            self.labels.append(label)
        classes, idx = np.unique(self.labels, return_inverse= True)
        self.y = torch.tensor(idx, dtype= torch.long)
        self.classes = classes
    
    def __len__(self):
        """
        Returns the number of images in the dataset.
        """
        return len(self.X)
    
    def __getitem__(self, idx):
        """
        Retrieves an image and its label at the specified index.
        
        Args:
            idx (int): Index of the image to retrieve.
        
        Returns:
            tuple: (image, label)
        """
        full_path = self.X[idx]
        image = Image.open(full_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        return (image, self.y[idx])

In [6]:
class CNN(nn.Module):
    """
    Define your CNN Model here 
    """
    def __init__(self, num_classes=10):
        """
        Initializes the layers of the CNN model.
        
        Args:
            num_classes (int): Number of output classes.
        """
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 
            
            nn.AdaptiveAvgPool2d((1, 1)) 
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3), 
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes) 
        )

    def forward(self, x):
        """
        Defines the forward pass of the model.
        
        Args:
            x (Tensor): Input tensor.
        
        Returns:
            Tensor: Output of the model.
        """
        x = self.features(x)
        x = self.classifier(x)
        return x

In [7]:
def train(model, train_loader, **kwargs):
    device = kwargs.get('device', torch.device("cpu"))
    val_loader = kwargs.get('val_loader')
    num_epochs = kwargs.get('num_epochs', 30)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2)
    
    best_acc = 0.0
    os.makedirs('submission', exist_ok=True) 

    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        val_acc = test(model, val_loader, device=device)
        scheduler.step(val_acc)
        print(f"Epoch {epoch+1} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), './submission/model.pt')
            print(f"--> Saved Best Model ({best_acc:.2f}%)")

In [8]:
def test(model, loader, **kwargs):
    device = kwargs.get('device', torch.device("cpu"))
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

In [9]:
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--train_dir', type=str, default='/Users/gaziaaziz/Documents/GitHub/Pytorch/hw4-data')
    parser.add_argument('--seed', type=int, default=42)
    args, _ = parser.parse_known_args()
    return args

In [10]:
def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")

    full_dataset = SUN397Dataset(data_dir=args.train_dir)
    train_size = int(0.8 * len(full_dataset))
    train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, len(full_dataset)-train_size])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = CNN(num_classes=len(full_dataset.classes)).to(device)
    train(model, train_loader, val_loader=val_loader, device=device, num_epochs=30)


In [ ]:
if __name__ == "__main__":
    main()

Using device: mps
Epoch 1 | Val Acc: 53.02%
--> Saved Best Model (53.02%)
Epoch 2 | Val Acc: 53.43%
--> Saved Best Model (53.43%)
Epoch 3 | Val Acc: 53.60%
--> Saved Best Model (53.60%)
Epoch 4 | Val Acc: 55.42%
--> Saved Best Model (55.42%)
Epoch 5 | Val Acc: 48.97%
